# 05 — LangGraph Agent

## About

**Purpose:** Build a LangGraph agent that routes a user question to either the RAG pipeline (Q&A over LEIE exclusions) or the XGBoost risk-scoring model, using an **LLM router** that classifies intent and extracts the provider NPI.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-05-16<br>
**Notes:** Loads artifacts produced by notebooks 03 (XGBoost model via MLflow) and 04 (Qdrant collection). Connects to the self-hosted Qdrant collection `leie_exclusions` (Docker, port 6333) built in 04 — **the Qdrant container must be running**. Gemini is called both inside the RAG tool and in the router (intent classification + NPI extraction) — re-running the tests in section 8 hits the API. Risk scoring does a **real NPI lookup** against the 500K-provider labelled dataset and applies the same encoding as the serving app.<br>
**Description:** Connects to the Qdrant vector store from 04 and the XGBoost model logged by 03 via MLflow, plus the provider lookup table and target-encoding maps. Defines two tools: `query_leie_rag` (Qdrant retriever + Gemini answer) and `score_provider_risk` (NPI → feature lookup → XGBoost probability). Builds a `StateGraph` with an LLM `router` node (Gemini classifies intent + extracts NPI) and a tool executor node. Compiles and tests both routing paths end-to-end.

### Change Control

| Date       | Version | Author      | Changes                                                                                          |
|------------|---------|-------------|-------------------------------------------------------------------------------------------------|
| 2026-05-16 | 1.0     | Ganapathy K | Initial version (keyword router, hardcoded-NPI risk stub)                                         |
| 2026-06-24 | 2.0     | Ganapathy K | LLM-based router (intent classify + NPI extraction); risk tool does real NPI lookup + scoring     |
| 2026-06-25 | 3.0     | Ganapathy K | Swap FAISS for the self-hosted Qdrant collection from 04 — single vector store across the pipeline |

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Setup
### 1.1 Imports

In [2]:
import warnings
import os
import logging
from datetime import datetime
from pathlib import Path
from typing import TypedDict

import pandas as pd
from dotenv import load_dotenv

from google import genai

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, END

import mlflow.xgboost

warnings.filterwarnings("ignore")

D:\Data Science\Visual Studio Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Configure logging

In [3]:
log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)
log_filename = log_folder / f"run_{datetime.now().strftime('%Y-%m-%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler(),
    ],
    force=True,
)
logger = logging.getLogger(__name__)

### 1.3 Config

In [4]:
qdrant_url = "http://localhost:6333"
qdrant_collection_name = "leie_exclusions"
mlflow_model_path = "../notebooks/mlruns/0/models/m-787607a155fb46b29bc9fcd13f990822/artifacts"
mlflow_tracking_uri = Path("../notebooks/mlruns").resolve().as_uri()

embedding_model_name = "all-MiniLM-L6-v2"
gemini_model_name = "gemini-2.5-flash"
retriever_k = 3

google_api_key_env_var = "GOOGLE_API_KEY_HEALTHCARE_PROVIDER_TERMINATION"

# Risk-scoring lookup: raw labelled dataset + target-encoding maps (shared with the serving app)
provider_lookup_path = Path("../data/processed/labelled_dataset.parquet")
encoding_maps_path = Path("../serving/encoding_maps.json")
risk_threshold = 0.5

# 16 model features, in the exact training/serving order
feature_columns = [
    "Entity Type Code",
    "Provider Business Mailing Address State Name",
    "Provider Business Mailing Address Telephone Number",
    "Provider Business Practice Location Address State Name",
    "Healthcare Provider Taxonomy Code_1",
    "Provider License Number State Code_1",
    "Provider Enumeration Year",
    "Last Update Year",
    "Provider Sex Code_F", "Provider Sex Code_M", "Provider Sex Code_U",
    "Healthcare Provider Primary Taxonomy Switch_1_N", "Healthcare Provider Primary Taxonomy Switch_1_Y",
    "Is Sole Proprietor_N", "Is Sole Proprietor_X", "Is Sole Proprietor_Y",
]

### 1.4 Authentication

In [5]:
load_dotenv()
google_api_key = os.getenv(google_api_key_env_var)
client = genai.Client(api_key=google_api_key)
logger.info(f"Gemini client ready — API key loaded: {google_api_key is not None}")

2026-07-02 15:55:24,811 | INFO | Gemini client ready — API key loaded: True


## 2. Connect to Qdrant Vector Store

Reconnects to the `leie_exclusions` collection built and persisted by notebook 04 (self-hosted Qdrant, Docker, port 6333). No re-embedding — the 8,482 vectors live in the Docker volume. **The Qdrant container must be running.**

In [6]:
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

qdrant_client = QdrantClient(url=qdrant_url)
vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=qdrant_collection_name,
    embedding=embeddings,
)

collection_info = qdrant_client.get_collection(qdrant_collection_name)
logger.info(f"Qdrant collection '{qdrant_collection_name}' connected — {collection_info.points_count} vectors")

2026-07-02 15:55:36,005 | INFO | No device provided, using cpu


2026-07-02 15:55:36,482 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-07-02 15:55:36,491 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


2026-07-02 15:55:36,719 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-07-02 15:55:36,720 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


...

2026-07-02 15:55:41,019 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


2026-07-02 15:55:41,257 | INFO | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


2026-07-02 15:55:41,495 | INFO | HTTP Request: GET http://localhost:6333/collections/leie_exclusions "HTTP/1.1 200 OK"


2026-07-02 15:55:41,641 | INFO | HTTP Request: GET http://localhost:6333/collections/leie_exclusions "HTTP/1.1 200 OK"


2026-07-02 15:55:41,642 | INFO | Qdrant collection 'leie_exclusions' connected — 8482 vectors


## 3. Load XGBoost Model

In [7]:
mlflow.set_tracking_uri(mlflow_tracking_uri)
model = mlflow.xgboost.load_model(mlflow_model_path)
logger.info(f"XGBoost model loaded from: {mlflow_model_path}")

2026-07-02 15:55:41,705 | INFO | HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


2026-07-02 15:55:42,146 | INFO | XGBoost model loaded from: ../notebooks/mlruns/0/models/m-787607a155fb46b29bc9fcd13f990822/artifacts


## 3b. Load Provider Lookup Table & Encoding Maps

For the risk tool to score a **real** NPI, load the raw labelled provider dataset (NPI → raw NPPES fields) and the target-encoding maps written by notebook 03. `encode_row` turns one raw record into the 16 model features using the exact same scheme as the serving app, so notebook and API scores agree.

In [8]:
import json

with open(encoding_maps_path) as file:
    encoding_maps = json.load(file)

lookup_columns = [
    "NPI", "Entity Type Code", "Provider Business Mailing Address Telephone Number",
    "Provider Enumeration Date", "Last Update Date", "Provider Sex Code",
    "Healthcare Provider Primary Taxonomy Switch_1", "Is Sole Proprietor",
    "Healthcare Provider Taxonomy Code_1", "Provider Business Mailing Address State Name",
    "Provider Business Practice Location Address State Name", "Provider License Number State Code_1",
]
provider_lookup = pd.read_parquet(provider_lookup_path, columns=lookup_columns)
logger.info(f"Provider lookup loaded — {len(provider_lookup):,} providers; encoding maps for {len(encoding_maps)} columns")


def encode_row(record: pd.Series) -> pd.DataFrame:
    """Encode one raw NPPES provider record into the 16 model features (same scheme as the serving app)."""
    def target_encode(column, value):
        return encoding_maps[column].get(str(value), 0.0)

    row = {
        "Entity Type Code": int(record["Entity Type Code"]) if pd.notna(record["Entity Type Code"]) else 0,
        "Provider Business Mailing Address State Name": target_encode("Provider Business Mailing Address State Name", record["Provider Business Mailing Address State Name"]),
        "Provider Business Mailing Address Telephone Number": float(record["Provider Business Mailing Address Telephone Number"]) if pd.notna(record["Provider Business Mailing Address Telephone Number"]) else 0.0,
        "Provider Business Practice Location Address State Name": target_encode("Provider Business Practice Location Address State Name", record["Provider Business Practice Location Address State Name"]),
        "Healthcare Provider Taxonomy Code_1": target_encode("Healthcare Provider Taxonomy Code_1", record["Healthcare Provider Taxonomy Code_1"]),
        "Provider License Number State Code_1": target_encode("Provider License Number State Code_1", record["Provider License Number State Code_1"]),
        "Provider Enumeration Year": pd.to_datetime(record["Provider Enumeration Date"]).year,
        "Last Update Year": pd.to_datetime(record["Last Update Date"]).year,
        "Provider Sex Code_F": int(record["Provider Sex Code"] == "F"),
        "Provider Sex Code_M": int(record["Provider Sex Code"] == "M"),
        "Provider Sex Code_U": int(record["Provider Sex Code"] == "U"),
        "Healthcare Provider Primary Taxonomy Switch_1_N": int(record["Healthcare Provider Primary Taxonomy Switch_1"] == "N"),
        "Healthcare Provider Primary Taxonomy Switch_1_Y": int(record["Healthcare Provider Primary Taxonomy Switch_1"] == "Y"),
        "Is Sole Proprietor_N": int(record["Is Sole Proprietor"] == "N"),
        "Is Sole Proprietor_X": int(record["Is Sole Proprietor"] == "X"),
        "Is Sole Proprietor_Y": int(record["Is Sole Proprietor"] == "Y"),
    }
    return pd.DataFrame([row], columns=feature_columns)

2026-07-02 15:55:42,699 | INFO | Provider lookup loaded — 500,000 providers; encoding maps for 4 columns


## 4. Define Tools

In [9]:
def query_leie_rag(question: str) -> str:
    retriever = vector_store.as_retriever(search_kwargs={"k": retriever_k})
    retrieved_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"Based on the following records:\n{context}\n\nAnswer this question: {question}"
    response = client.models.generate_content(model=gemini_model_name, contents=prompt)
    return response.text


def score_provider_risk(npi) -> str:
    if npi in (None, "", "null"):
        return "No NPI was supplied, so I can't score a specific provider. Please include a 10-digit NPI."
    try:
        npi_value = int(str(npi).strip())
    except ValueError:
        return f"'{npi}' is not a valid NPI (expected 10 digits)."
    matches = provider_lookup[provider_lookup["NPI"] == npi_value]
    if matches.empty:
        return f"NPI {npi_value} was not found in the provider dataset."
    features = encode_row(matches.iloc[0])
    probability = float(model.predict_proba(features)[0][1])
    tier = "high" if probability >= risk_threshold else "low"
    return f"Risk score for NPI {npi_value}: {probability:.4f} (risk tier: {tier} — higher means more likely to be excluded)."


logger.info("Tools defined: query_leie_rag, score_provider_risk (real NPI lookup)")

2026-07-02 15:55:42,804 | INFO | Tools defined: query_leie_rag, score_provider_risk (real NPI lookup)


## 5. Define Agent State

In [10]:
class AgentState(TypedDict):
    question: str
    tool_used: str
    npi: str
    answer: str


logger.info("Agent state ready")

2026-07-02 15:55:42,900 | INFO | Agent state ready


## 6. Define Graph Nodes

In [11]:
import re


def classify_intent(question: str) -> dict:
    """LLM router: classify intent (risk vs rag) and extract any NPI from the question."""
    prompt = (
        "You are the router for a healthcare provider-integrity assistant. "
        "Classify the question into exactly one intent and extract any NPI (a 10-digit provider identifier).\n\n"
        'intent = "risk": the user wants the ML model\'s exclusion-RISK SCORE or probability for ONE specific '
        'provider (usually given by an NPI). Signals: "risk score", "how risky", "probability", '
        '"should we pay/worry about <NPI>", or a bare 10-digit number.\n'
        'intent = "rag": the user asks about the exclusion RECORDS themselves — looking up, listing, counting, '
        'or explaining who is excluded, why, where, or when. Signals: "are there any", "who", "list", '
        '"how many", "what reason", "in <state>".\n\n'
        'Respond with ONLY a JSON object: {"intent": "risk" | "rag", "npi": "<digits or null>"}.\n\n'
        "Examples:\n"
        'Q: Are there any excluded providers in Texas? -> {"intent": "rag", "npi": null}\n'
        'Q: How many providers were excluded for fraud? -> {"intent": "rag", "npi": null}\n'
        'Q: What is the risk score for NPI 1234567890? -> {"intent": "risk", "npi": "1234567890"}\n'
        'Q: Should we be worried about 1679576722? -> {"intent": "risk", "npi": "1679576722"}\n\n'
        f"Question: {question}"
    )
    response = client.models.generate_content(model=gemini_model_name, contents=prompt)
    match = re.search(r"\{.*\}", response.text, re.DOTALL)
    parsed = json.loads(match.group(0)) if match else {"intent": "rag", "npi": None}
    intent = "risk" if parsed.get("intent") == "risk" else "rag"
    npi = parsed.get("npi")
    npi = "" if npi in (None, "null", "") else str(npi).strip()
    return {"intent": intent, "npi": npi}


def router_node(state: AgentState) -> AgentState:
    decision = classify_intent(state["question"])
    state["tool_used"] = "score_provider_risk" if decision["intent"] == "risk" else "query_leie_rag"
    state["npi"] = decision["npi"]
    logger.info(f"Router → {state['tool_used']} | npi={state['npi'] or 'none'}")
    return state


def tool_node(state: AgentState) -> AgentState:
    if state["tool_used"] == "score_provider_risk":
        state["answer"] = score_provider_risk(state["npi"])
    else:
        state["answer"] = query_leie_rag(state["question"])
    return state


logger.info("Nodes defined: router_node (LLM-based), tool_node")

2026-07-02 15:55:42,996 | INFO | Nodes defined: router_node (LLM-based), tool_node


## 7. Build and Compile Graph

In [12]:
graph = StateGraph(AgentState)

graph.add_node("router", router_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("router")
graph.add_edge("router", "tool")
graph.add_edge("tool", END)

agent = graph.compile()
logger.info("Agent compiled successfully")

2026-07-02 15:55:43,092 | INFO | Agent compiled successfully


## 8. Test
### 8.1 RAG path

In [13]:
result = agent.invoke({"question": "Are there any excluded providers in Texas?", "tool_used": "", "npi": "", "answer": ""})
print("Tool used:", result["tool_used"])
print("Answer:", result["answer"])

2026-07-02 15:55:43,203 | INFO | AFC is enabled with max remote calls: 10.


2026-07-02 15:55:44,649 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


2026-07-02 15:55:44,651 | INFO | Router → query_leie_rag | npi=none


2026-07-02 15:55:44,688 | INFO | HTTP Request: POST http://localhost:6333/collections/leie_exclusions/points/query "HTTP/1.1 200 OK"


2026-07-02 15:55:44,692 | INFO | AFC is enabled with max remote calls: 10.


2026-07-02 15:55:46,301 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Tool used: query_leie_rag
Answer: Yes, based on the records provided, there are excluded providers in Texas.


### 8.2 Risk scoring path

In [14]:
result_2 = agent.invoke({"question": "What is the exclusion risk score for provider NPI 1871596098?", "tool_used": "", "npi": "", "answer": ""})
print("Tool used:", result_2["tool_used"])
print("NPI extracted:", result_2["npi"])
print("Answer:", result_2["answer"])

2026-07-02 15:55:46,408 | INFO | AFC is enabled with max remote calls: 10.


2026-07-02 15:55:47,831 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


2026-07-02 15:55:47,832 | INFO | Router → score_provider_risk | npi=1871596098


Tool used: score_provider_risk
NPI extracted: 1871596098
Answer: Risk score for NPI 1871596098: 0.0760 (risk tier: low — higher means more likely to be excluded).


## 9. Summary

**Pitch:** Built a LangGraph agent that routes a user question to one of two tools — a RAG pipeline for exclusion Q&A or an XGBoost classifier for provider risk scoring — using an LLM router that classifies intent and extracts the NPI.

### Approach
- Two tools: `query_leie_rag` (self-hosted Qdrant + Gemini, top-3 retrieval) and `score_provider_risk` (real NPI → feature lookup against the 500K labelled dataset → XGBoost probability, encoded identically to the serving app)
- `StateGraph` with two nodes: `router` (Gemini classifies intent + extracts NPI into the agent state) and `tool` (executes the chosen tool with that NPI)
- Single vector store (`leie_exclusions` Qdrant collection) shared with notebook 04 — no FAISS in the pipeline
- Both routing paths verified end-to-end

### Tested
- "Are there any excluded providers in Texas?" → routed to `query_leie_rag`
- "What is the exclusion risk score for provider NPI 1871596098?" → routed to `score_provider_risk`, NPI extracted and scored

### Limitations
- Router is a single Gemini call returning JSON; malformed output falls back to the RAG path
- Risk lookup is bounded to NPIs present in the 500K labelled sample (a production system would query the full NPPES registry)

### Next iteration
- Add RBAC role-filtering (Qdrant payload filter) + a cross-encoder reranker to the RAG tool (capstone SHOULD tier)
- RAGAS eval harness over a golden Q&A set; structured trajectory/observability trace
- Expose both tools over an MCP server